Inainte sa trimiti notebook-ul:

* asigura-te ca totul se executa cum trebuie. Prima oara, **restarteaza kernel-ul** (in meniu, selecteaza Kernel$\rightarrow$Restart & Run all);

* nu lasa niciun mesaj la output (`disp`, `printf`, linii care nu sunt incheiate cu `;`) in functiile pe care le scrii decat daca ti se specifica explicit sa faci acest lucru! Orice abatere va fi **depunctata automat** de grader;

* citeste instructiunile de [aici](https://github.com/radusqrt/numerical-methods/blob/master/lab_instructions.md) cu atentie;

* asigura-te ca rezolvi toate celulele in care scrie `YOUR CODE HERE` sau `YOUR ANSWER HERE` si ca iti treci numele mai jos:

In [1]:
STUDENT_NAME = "Andrei Ionescu";

---

# Laborator 6: Soluții ale ecuațiilor neliniare
### Autor: Radu Stochitoiu

## Problema 1

Implementati funcția ```bisection(f, a, b, tol, max_steps)``` ce primește un handle la o funcție $f$ (se poate evalua cu ajutorul funcției ```feval```) continuă cu valori reale, capetele unui interval $a$ și $b$, o toleranță sub care se poate opri algoritmul $tol$ și un număr maxim de pași $max\_steps$ și returnează o rădăcină a lui $f$ aproximată după $steps$ pași, $r$, eroarea de la ultimul pas $err$ și numărul de pași efectuați $steps$.

In [2]:
function [r, err, steps] = bisection(f, a, b, tol, max_steps)
    steps = 0;
    
    while 1
        r = (a+b) / 2;
        if feval(f, a) * feval(f, r) < 0
            b = r;
        else
            a = r;
        endif
        
        err = b - a;
        if (++steps == max_steps || err < tol)
            return;
        endif
    endwhile
endfunction

In [3]:
function [res] = f(x)
    res = x^3 + cos(x) + e^x + sqrt(x + 1);
endfunction

[r, err, steps] = bisection(@f, -1, 100, 1e-8, 100);
assert(abs(f(r)) < 1e-5);
assert(steps < 40);

## Problema 2

Implementați funcția ```newton_raphson(f, df, x0, tol, max_steps)``` ce primește un handle la o funcție $f$ (se poate evalua cu ajutorul funcției ```feval```) continuă cu valori reale, un handle la derivata lui $f$, $df$, o aproximare inițială a soluției $x_0$, o toleranță sub care se poate opri algoritmul $tol$ și un număr maxim de pași $max\_steps$ și returnează o rădăcină a lui $f$ aproximată după $steps$ pași, $r$, eroarea de la ultimul pas $err$ și numărul de pași efectuați $steps$.

In [4]:
function [r, err, steps] = newton_raphson(f, df, x0, tol, max_steps)
    steps = 0;
    r = x0;
    
    while 1
        old_r = r;
        r = r - feval(f, r) / feval(df, r);
        
        err = abs(old_r - r);
        if (++steps == max_steps || err < tol)
            return;
        endif
    endwhile
endfunction

In [5]:
function [res] = f(x)
    res = x^2 - 3;
endfunction

function [res] = df(x)
    res = 2 * x;
endfunction

[r, err, steps] = newton_raphson(@f, @df, 1.7, 1e-8, 100);
assert(f(r) < 1e-10);
assert(steps < 5);

[r, err, steps] = newton_raphson(@f, @df, 31231, 1e-8, 100);
assert(f(r) < 1e-10);
assert(steps < 25);


## Problema 3

Implementați funcția ```secant(f, x0, x1, tol, max_steps)``` ce primește un handle la o funcție $f$ (se poate evalua cu ajutorul funcției ```feval```) continuă cu valori reale, două aproximări inițiale ale soluției $x_0$ și $x_1$, o toleranță sub care se poate opri algoritmul $tol$ și un număr maxim de pași $max\_steps$ și returnează o rădăcină a lui $f$ aproximată după $steps$ pași, $r$, eroarea de la ultimul pas $err$ și numărul de pași efectuați $steps$.

In [6]:
function [r, err, steps] = secant(f, x0, x1, tol, max_steps)
    steps = 0;
    % x - aproximarea veche; r - arpoximarea noua
    r = x1;
    x = x0;
    
    while 1
        old_r = r;
        r = r - (x-r) / (feval(f, x) - feval(f, r)) * feval(f, r);
        x = old_r;
        
        err = abs(r - old_r);
        if (++steps == max_steps || err < tol)
            return;
        endif
    endwhile
endfunction

In [7]:
function [res] = f(x)
    res = x^2 - 3;
endfunction

[r, err, steps] = secant(@f, 1.65, 1.7, 1e-8, 100);
assert(f(r) < 1e-10);
assert(steps < 5);

[r, err, steps] = secant(@f, 312312, 1000, 1e-8, 100);
assert(f(r) < 1e-10);
assert(steps < 25);

## Problema 4

Implementați funcția ```newton(F_handles, x0, tol, max_steps, h)``` ce primește un cell array ce conține un număr variabil de handles la funcții $f_i$ (se pot evalua cu ajutorul funcției ```feval```) continue cu valori reale, un vector $x_0$ cu aproximarea inițială a sistemului de ecuații neliniare $F(x) = 0$, o toleranță sub care se poate opri algoritmul $tol$, un număr maxim de pași $max\_steps$ și o variabilă ce va ajuta la aproximarea valorilor matricei jacobiene $h$ și returnează un vector $r$ de rădăcini ale sistemului neliniar $F(x) = 0$.

Aproximarea lui $\frac{\partial f_i}{\partial x_j}$ se face folosind metoda diferențelor finite (preferabil a doua din laborator).

In [8]:
function [r, err, steps] = secant(F_handles, x0, tol, max_steps, h)
    steps = 0;
    r = x0;
    n = length(r);
    DF = zeros(n);
    FF = zeros(n, 1);
    
    while 1
        % Gasim DF pentru aproximarile x curente
        % Pentru fiecare f(i)
        for i = 1:n
            f = F_handles{i};
            % Derivata dupa x(j)
            for j = 1:n
                f_plus_h = feval(f, [r(1:j-1), r(j)+h, r(j+1:n)]);
                f_minus_h = feval(f, [r(1:j-1), r(j)-h, r(j+1:n)]);
                DF(i, j) = 1/(2*h) * (f_plus_h - f_minus_h);
            endfor
        endfor
        
        % Aflam -F(x)
        for i = 1:n
            FF(i) = -feval(F_handles{i}, r);
        endfor
        
        % Factorizam DF in QR ca sa aflam Dx
        [Q, R] = Householder(DF);
        y = Q' * FF;
        Dx = SST(R, y);
        
        old_x = r;
        r = r + Dx;
        
        err = norm(r - old_x);
        if (++steps == max_steps || err < tol)
            return;
        endif
    endwhile
endfunction

% SST pentru a rezolva R*x=b, unde R e superior triu.
function [x] = SST(A, b)
    [n, m] = size(A);
    x = b(m) / A(m,m);
    for i = m-1:-1:1
        x = [(b(i) - A(i, i+1:m) * x) / A(i,i) ; x];
    endfor
endfunction

% Factorizare QR Householder
function [Q R] = Householder(A)
    [m n] = size(A);
    Q = eye(m);
    
    for i = 1 : min(m-1, n)
        v = zeros(m, 1);
        sgn = sign(A(i, i));
        if (sgn == 0)
            sgn = 1;
        endif
        
        v(i) = A(i, i) + sgn * norm(A(i : m, i)');
        v(i+1 : m) = A(i+1 : m, i);
        
        H = eye(m) - 2 * v * v' / (v' * v);
        
        A = H * A;
        Q = H * Q;
    endfor
    
    Q = Q';
    R = A;
endfunction


In [9]:
function [res] = f1(x)
    res = x(1) + 2 * x(2) - 2;
endfunction

function [res] = f2(x)
    res = x(1)^2 + 4 * x(2)^2 - 4;
endfunction

[R, err, steps] = secant({@f1; @f2}, rand(2, 1), 1e-5, 100, 1e-5);
assert(norm(f1(R)) < 1e-10);
assert(norm(f1(R)) < 1e-10);
assert(steps < 10);